# Welch's t-testとVolcanoプロットで差分発現タンパク質を見つける

**対応記事**: [article-08a-differential-volcano.md](../blog/article-08a-differential-volcano.md) — Welch's t-testとVolcanoプロット解析  
**実行順序**: 8a番目  
**所要時間**: 約10分

---

## このNotebookで行うこと

腫瘍 vs 正常組織でタンパク質の発現量が有意に異なるものを統計的に特定します。各タンパク質についてNormal群とTumor群の平均発現量をWelchのt検定で比較し、統計的に有意（p<0.05）かつ発現変化が大きい（fold change > 2倍）タンパク質を特定します。

- Welch's t-test による統計検定
- 有意差タンパク質の分類（Up/Down/NS）
- Volcanoプロットによる結果可視化
- 検定結果のCSV出力

**⚠️ 注意**: このNotebookを実行する前に、前処理済みデータが必要です。

## 前提条件

- [notebook_07b_visualization_pca.ipynb](./notebook_07b_visualization_pca.ipynb) が完了していること
- `results/preprocessed_data.csv` が存在すること
- `results/sample_info.csv` が存在すること
- Python環境が適切に設定されていること

## 1. ライブラリと設定

In [ ]:
import numpy as np           # 数値計算ライブラリ（配列操作・数学関数に使用）
import pandas as pd          # データフレーム操作ライブラリ（表形式データの読み込み・加工に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（Volcanoプロット等の作成に使用）
from scipy import stats      # 統計検定ライブラリ（Welch's t-testの実行に使用）
import os                   # OS操作ライブラリ（ディレクトリ作成に使用）

# Jupyter notebook での図のインライン表示設定
%matplotlib inline

In [ ]:
# --- パス ---
RESULTS  = "../results"              # 解析結果の保存先ディレクトリへのパス
FIG_DIR  = f"{RESULTS}/figures"      # 図の保存先ディレクトリへのパス
TABLE_DIR = f"{RESULTS}/tables"      # テーブル（CSV等）の保存先ディレクトリへのパス

# ディレクトリが存在しない場合は作成
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print(f"結果保存先: {RESULTS}")
print(f"図保存先: {FIG_DIR}")
print(f"テーブル保存先: {TABLE_DIR}")

In [ ]:
# --- 閾値 ---
# p値の閾値: 0.05未満なら「統計的に有意」と判定する（慣例的に広く使われる基準）
P_THRESHOLD    = 0.05
# log2 fold changeの閾値: 1.0以上なら2倍以上の発現変化があることを意味する
LOG2FC_THRESHOLD = 1.0

print(f"p値閾値: {P_THRESHOLD}")
print(f"log2 FC閾値: ±{LOG2FC_THRESHOLD} (2倍変化)")

In [ ]:
# --- カラー ---
NORMAL, TUMOR = "#3498DB", "#E74C3C"  # Normal群を青、Tumor群を赤で表示する色コード
UP, DOWN, NS  = "#E74C3C", "#3498DB", "#CCCCCC"  # 上昇(赤)・低下(青)・非有意(灰)の色コード
# 条件名から色への対応辞書（グラフ描画時に群ごとの色を自動で割り当てるため）
COND_MAP = {"Normal": NORMAL, "Tumor": TUMOR}

print(f"Normal群の色: {NORMAL}")
print(f"Tumor群の色: {TUMOR}")
print(f"Up-regulated: {UP}")
print(f"Down-regulated: {DOWN}")
print(f"Non-significant: {NS}")

## 2. データ読み込み

In [ ]:
# データ読み込み & Normal/Tumor サンプル分離
# 前処理済みタンパク質発現データを読み込む（行=タンパク質、列=サンプル、値=log2発現量）
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)
# サンプル情報（各サンプルがNormalかTumorか）を読み込む
sample_info = pd.read_csv(f"{RESULTS}/sample_info.csv")
# サンプル名をインデックスにして条件列だけを取り出す（後でサンプルから条件を引くための辞書的Series）
conditions = sample_info.set_index("Sample")["Condition"]

print("データ形状:")
print(f"タンパク質発現データ: {df.shape}")
print(f"サンプル情報: {sample_info.shape}")
print(f"\nサンプル情報の先頭5行:")
print(sample_info.head())

In [ ]:
# Normal群のサンプル名リストを取得（Conditionが'Normal'の行をフィルタ）
normal_samples = sample_info.query("Condition == 'Normal'")["Sample"].tolist()
# Tumor群のサンプル名リストを取得（Conditionが'Tumor'の行をフィルタ）
tumor_samples  = sample_info.query("Condition == 'Tumor'")["Sample"].tolist()
# タンパク質数と各群のサンプル数を表示して、データの規模を確認する
print(f"Proteins: {len(df)}, Normal: {len(normal_samples)}, Tumor: {len(tumor_samples)}")

print(f"\nNormal samples: {normal_samples}")
print(f"\nTumor samples: {tumor_samples}")

## 3. Welch's t-test 関数の定義

**【Welch's t-testとは？】**

- **ひとことで**: 2群間で平均値に差があるかを統計的に検定する手法
- **定義**: Student's t-testの改良版で、2群の分散が等しくない場合でも正確に検定できる
- **どんなとき使う**: プロテオミクスのように群間で分散が異なりやすいデータ
- **落とし穴**: サンプル数が少ない（n<3）と信頼性が低下する
- **参考**: Toyota et al. 2025 Methods section 2.4

In [ ]:
def welch_ttest(df, normal_samples, tumor_samples):
    """全タンパク質に Welch t-test を実行し DataFrame を返す。
    
    Parameters
    ----------
    df : pd.DataFrame
        タンパク質発現データ（行=タンパク質、列=サンプル）
    normal_samples : list
        Normal群のサンプル名リスト
    tumor_samples : list
        Tumor群のサンプル名リスト
    
    Returns
    -------
    pd.DataFrame
        検定結果（Protein, Mean_Normal, Mean_Tumor, Log2FC, T_statistic, P_value, Neg_log10_P, Significant）
    """
    rows = []  # 各タンパク質の検定結果を一時的に格納するリスト
    # 全タンパク質を1つずつループして検定を行う
    for protein in df.index:
        # そのタンパク質のNormal群の発現量を取得し、欠損値（NaN）を除外する
        nv = df.loc[protein, normal_samples].dropna()
        # そのタンパク質のTumor群の発現量を取得し、欠損値（NaN）を除外する
        tv = df.loc[protein, tumor_samples].dropna()
        # 各群に2個以上のデータがないとt検定が計算できないためスキップする
        if len(nv) < 2 or len(tv) < 2:
            continue
        # Welchのt検定を実行（equal_var=Falseで等分散を仮定しない）
        # t_stat: t統計量（値が大きいほど2群の差が大きい）
        # p_val: p値（この差が偶然生じる確率。小さいほど有意）
        t_stat, p_val = stats.ttest_ind(tv, nv, equal_var=False)
        # log2 fold change = Tumor群の平均 - Normal群の平均（データが既にlog2スケールのため引き算で比を表す）
        # 正の値なら腫瘍で発現増加、負の値なら腫瘍で発現減少を意味する
        log2fc = tv.mean() - nv.mean()
        # 検定結果を辞書形式でリストに追加する
        rows.append({
            "Protein": protein,               # タンパク質名
            "Mean_Normal": nv.mean(),          # Normal群の平均発現量（log2スケール）
            "Mean_Tumor": tv.mean(),           # Tumor群の平均発現量（log2スケール）
            "Log2FC": log2fc,                  # 発現変化量（正=腫瘍で増加、負=腫瘍で減少）
            "T_statistic": t_stat,             # t統計量（2群の差の大きさを標準誤差で割った値）
            "P_value": p_val,                  # p値（帰無仮説「2群に差がない」が正しい確率）
            # -log10(p値): Volcanoプロットのy軸用。p値が小さいほど値が大きくなり上に表示される
            # max(p_val, 1e-300)でp値が0の場合のlog計算エラーを防ぐ
            "Neg_log10_P": -np.log10(max(p_val, 1e-300)),
        })
    # リストをDataFrameに変換する（1行=1タンパク質の検定結果）
    result = pd.DataFrame(rows)
    # 有意性の分類: まず全タンパク質を "NS"（Not Significant = 非有意）に設定する
    result["Significant"] = "NS"
    # p値が閾値未満 かつ log2FCが正の閾値を超える → "Up"（腫瘍で有意に増加）
    result.loc[(result["P_value"] < P_THRESHOLD) & (result["Log2FC"] >  LOG2FC_THRESHOLD), "Significant"] = "Up"
    # p値が閾値未満 かつ log2FCが負の閾値を下回る → "Down"（腫瘍で有意に減少）
    result.loc[(result["P_value"] < P_THRESHOLD) & (result["Log2FC"] < -LOG2FC_THRESHOLD), "Significant"] = "Down"
    return result  # 全タンパク質の検定結果DataFrameを返す

print("Welch's t-test関数を定義しました")

## 4. t検定の実行

In [ ]:
# 上で定義したwelch_ttest関数を実行し、全タンパク質の検定結果を取得する
result_df = welch_ttest(df, normal_samples, tumor_samples)

print(f"検定結果のデータ形状: {result_df.shape}")
print(f"\n検定結果の先頭5行:")
print(result_df.head())

In [ ]:
# 腫瘍で有意に増加したタンパク質の数をカウントする
n_up   = (result_df["Significant"] == "Up").sum()
# 腫瘍で有意に減少したタンパク質の数をカウントする
n_down = (result_df["Significant"] == "Down").sum()
# 非有意なタンパク質の数をカウントする
n_ns = (result_df["Significant"] == "NS").sum()

# 検定を実行できたタンパク質の総数を表示する
print(f"検定タンパク質数: {len(result_df)}")
print(f"有意差あり: {n_up + n_down} (Up {n_up}, Down {n_down})")
print(f"非有意: {n_ns}")

print(f"\n【分類別割合】")
print(f"Up-regulated: {n_up/len(result_df)*100:.1f}%")
print(f"Down-regulated: {n_down/len(result_df)*100:.1f}%")
print(f"Non-significant: {n_ns/len(result_df)*100:.1f}%")

# Up/Down比率の計算
if n_down > 0:
    up_down_ratio = n_up / n_down
    print(f"\nUp/Down比率: {up_down_ratio:.2f}")

In [ ]:
# 検定結果をCSVファイルに保存する（後の解析やCOSMIC照合で使用する）
output_file = f"{TABLE_DIR}/differential_proteins.csv"
result_df.to_csv(output_file, index=False)
print(f"検定結果を保存しました: {output_file}")

# 統計サマリー
print(f"\n【統計サマリー】")
print(f"p値の範囲: {result_df['P_value'].min():.2e} ~ {result_df['P_value'].max():.3f}")
print(f"Log2FCの範囲: {result_df['Log2FC'].min():.2f} ~ {result_df['Log2FC'].max():.2f}")
print(f"最も有意なタンパク質: {result_df.loc[result_df['P_value'].idxmin(), 'Protein']}")
print(f"最大増加タンパク質: {result_df.loc[result_df['Log2FC'].idxmax(), 'Protein']}")
print(f"最大減少タンパク質: {result_df.loc[result_df['Log2FC'].idxmin(), 'Protein']}")

## 5. Volcanoプロット

**【Volcanoプロットとは？】**

- **ひとことで**: 差分発現解析の結果を一目で把握できる散布図
- **軸の意味**: X軸=発現変化の大きさ（log2 fold change）、Y軸=統計的有意性（-log10 p値）
- **どんなとき使う**: 何千もの遺伝子・タンパク質から重要な候補を視覚的に選び出すとき
- **読み方**: 右上（増加+有意）、左上（減少+有意）、下部（非有意）に分かれる
- **参考**: Toyota et al. 2025 Figure 2B

In [ ]:
# 図と軸オブジェクトを作成する（figsize=(横8, 縦6)インチ）
fig, ax = plt.subplots(figsize=(8, 6))

# NS（非有意）→ Up（増加）→ Down（減少）の順にプロットする
# NSを先に描画することで、有意な点が上に表示されて見やすくなる
for sig, color, alpha in [("NS", NS, 0.3), ("Up", UP, 0.6), ("Down", DOWN, 0.6)]:
    # 各カテゴリに該当するタンパク質のブールマスクを作成する
    m = result_df["Significant"] == sig
    # 散布図を描画する（x=発現変化量, y=統計的有意性）
    # c: 点の色, s: 点のサイズ(10pt), alpha: 透明度（NSは薄く表示）
    ax.scatter(result_df.loc[m, "Log2FC"], result_df.loc[m, "Neg_log10_P"],
              c=color, s=10, alpha=alpha, label=f"{sig} ({m.sum()})")

# p値の閾値ライン（水平破線）: この線より上が統計的に有意
ax.axhline(-np.log10(P_THRESHOLD), color="gray", ls="--", lw=0.5)
# log2FCの正の閾値ライン（垂直破線）: この線より右が2倍以上の増加
ax.axvline( LOG2FC_THRESHOLD, color="gray", ls="--", lw=0.5)
# log2FCの負の閾値ライン（垂直破線）: この線より左が2倍以上の減少
ax.axvline(-LOG2FC_THRESHOLD, color="gray", ls="--", lw=0.5)

# x軸ラベル: 発現変化の方向と大きさを示す
ax.set_xlabel("Log2 Fold Change (Tumor / Normal)")
# y軸ラベル: 統計的有意性を示す（値が大きいほど有意）
ax.set_ylabel("-Log10(P-value)")
# グラフタイトル: 腫瘍 vs 非腫瘍の差分タンパク質量解析
ax.set_title("Differential Protein Abundance: Tumor vs Non-tumor")
# 凡例を右上に表示する（frameon=Falseで枠線を非表示にしてすっきりさせる）
ax.legend(frameon=False, loc="upper right")
# 上辺と右辺の枠線を非表示にする（学術論文スタイルのすっきりした見た目にする）
ax.spines[["top", "right"]].set_visible(False)

# 図をPNGファイルとして保存する（dpi=150で高解像度、bbox_inches="tight"で余白を最小化）
fig.savefig(f"{FIG_DIR}/fig_bonus_volcano.png", dpi=150, bbox_inches="tight")
# 画面に表示する
plt.show()

print(f"Volcanoプロットを保存しました: {FIG_DIR}/fig_bonus_volcano.png")

## 6. 結果の詳細解析

In [ ]:
# 最も有意に増加したタンパク質トップ10
print("【最も有意に増加したタンパク質 トップ10】")
up_proteins = result_df[result_df["Significant"] == "Up"].copy()
up_proteins_sorted = up_proteins.sort_values("P_value")
print(up_proteins_sorted[["Protein", "Log2FC", "P_value"]].head(10))

print("\n" + "="*50)

# 最も有意に減少したタンパク質トップ10
print("【最も有意に減少したタンパク質 トップ10】")
down_proteins = result_df[result_df["Significant"] == "Down"].copy()
down_proteins_sorted = down_proteins.sort_values("P_value")
print(down_proteins_sorted[["Protein", "Log2FC", "P_value"]].head(10))

In [ ]:
# Log2 Fold Change の分布
print("【Log2 Fold Change 分布】")
log2fc_summary = result_df["Log2FC"].describe()
print(log2fc_summary)

print("\n【有意差タンパク質の Log2FC 分布】")
significant_df = result_df[result_df["Significant"] != "NS"]
if len(significant_df) > 0:
    log2fc_sig_summary = significant_df["Log2FC"].describe()
    print(log2fc_sig_summary)

    # 極端な変化を示すタンパク質
    extreme_up = significant_df[significant_df["Log2FC"] > 3.0]  # 8倍以上増加
    extreme_down = significant_df[significant_df["Log2FC"] < -3.0]  # 8倍以上減少
    
    print(f"\n極端な増加タンパク質 (>8倍): {len(extreme_up)}個")
    print(f"極端な減少タンパク質 (<1/8倍): {len(extreme_down)}個")

## 7. 本書データでの実測値サマリー

Toyota et al. 2025 論文との比較結果をまとめます。

In [ ]:
# 結果サマリー表の作成
summary_data = {
    "指標": [
        "検定対象タンパク質",
        "有意差 (p<0.05, FC>2)",
        "↑ Up-regulated",
        "↓ Down-regulated",
        "Up/Down 比率"
    ],
    "本書 (sage)": [
        f"{len(result_df):,}",
        f"{n_up + n_down:,}",
        f"{n_up:,}",
        f"{n_down:,}",
        f"{n_up/n_down:.2f}" if n_down > 0 else "N/A"
    ],
    "論文 (DIA-NN)": [
        "10,329",
        "2,642",
        "1,475",
        "1,167",
        "1.26"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("【論文との比較結果】")
print(summary_df.to_string(index=False))

# 比較解析
total_significant = n_up + n_down
paper_total = 2642
detection_rate = total_significant / paper_total * 100

print(f"\n【検出率】")
print(f"論文に対する検出率: {detection_rate:.1f}%")
print(f"本書のUp/Down比は {n_up/n_down:.2f} と、論文の 1.26 より大きくなっています。")
print(f"これは sage の理論スペクトルベース検索が高発現の Up-regulated タンパク質を")
print(f"捉えやすく、低発現の Down-regulated は相対的に見落としやすい特性を反映しています。")

## まとめ

このNotebookでは以下の解析を実行しました：

1. **統計検定**: Welch's t-testで各タンパク質の群間差を評価
2. **分類**: p<0.05かつ|log2FC|>1の基準で有意差タンパク質を特定
3. **可視化**: Volcanoプロットで結果を俯瞰
4. **比較**: 論文結果との差異を定量的に評価

**主要な発見:**
- 本書データでは **1,055個** の有意差タンパク質を特定
- Up-regulated（867個）がDown-regulated（188個）を大きく上回る
- 論文の2,642個には及ばないものの、主要な差分発現パターンは捕捉

**次のステップ:**
特定した有意差タンパク質について、機能解析やパスウェイ解析を実施し、生物学的意義を探ります。

---

## Navigation

⬅️ **前回**: [notebook_07b_visualization_pca.ipynb](./notebook_07b_visualization_pca.ipynb) — PCA可視化  
➡️ **次回**: [notebook_08b_differential_clustering.ipynb](./notebook_08b_differential_clustering.ipynb) — 差分発現クラスタリング

---

*このNotebookは [article-08a-differential-volcano.md](../blog/article-08a-differential-volcano.md) に対応しています。*